---
## 5. Limitations: where Monte Carlo struggles
*Lead: Robin Levasseur*

TODO (Robin): 2-3 sentences introducing the part, and announce 5.1 to 5.5.

In [ ]:
# Setup: imports, plotting style and random seed
import sys
from pathlib import Path

ROOT = Path.cwd() if (Path.cwd() / "mcpert").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))  # makes the mcpert package importable

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from IPython.display import display

import mcpert as mc
from mcpert.examples import dinner, software_release
from mcpert.style import BLUE, ORANGE, AQUA, INK, MUTED, use_style

use_style()
SEED = 2026  # every number in this notebook is reproducible

In [ ]:
# Data used in this part
project, pert_model = software_release()                  # the 7-task running example (Part 3)
ratios = mc.load_sip()["ratio"].to_numpy()                # SiP ratios actual / estimate (Part 1)
sip_model = mc.EmpiricalRatio(project.durations, ratios)  # duration = planned x a real SiP ratio
plan = project.planned_duration()

### 5.1 How heavy is the tail of real overruns?

TODO (Robin): define a heavy tail of index $\alpha$ ($P(R > x) \approx C x^{-\alpha}$), explain that $\mathbb E[R^p] < \infty$
iff $p < \alpha$ (so no CLT when $\alpha \le 2$), give the Hill estimator
$1/\hat\alpha_k = \frac1k \sum_{i \le k} \ln(X_{(i)} / X_{(k+1)})$ and why it works for a Pareto law
($\ln(X/x_{\min})$ is exponential with rate $\alpha$).

In [ ]:
# Hill plot of the SiP ratios
# TODO (Robin): plot mc.hill_tail_index(ratios, k) for k in np.arange(20, 3000, 10),
# with a dashed horizontal line at alpha = 2.

TODO (Robin): what does the Hill plot say? Is $\alpha$ above or below 2? Why can a finite sample not
"prove" an infinite variance? Mention Flyvbjerg et al. (2022).

### 5.2 Slow and erratic convergence

TODO (Robin): one sentence describing the experiment (3 models x 5 runs).

In [ ]:
# Running estimate of E[T] for three delay models
# TODO (Robin): for pert_model, sip_model and mc.ParetoRatio(project.durations, alpha=1.5):
# 5 runs of n = 100,000 simulations each, plot mc.running_mean(T) against n (log x-axis),
# one panel per model (plt.subplots(1, 3)). Plot only ~400 log-spaced points per curve.

TODO (Robin): interpret the three panels. How fast does the error decrease when $\alpha = 1.5$?
(generalised CLT: like $n^{1/\alpha - 1}$ instead of $n^{-1/2}$).

### 5.3 Error bars that lie

TODO (Robin): describe the experiment: coverage of the 95% CLT interval for $\mathbb E[T]$ as $n$ grows, for the
three models; the Pareto case uses a 4-task chain where $\mathbb E[T]$ is known exactly; the quantile interval
(Proposition 5) is the control.

In [ ]:
# Coverage of 95% intervals as n grows
# TODO (Robin):
# 1. True values: chain = mc.Project([5, 10, 5, 1], [[], [0], [1], [2]]) with mc.ParetoRatio(..., alpha=1.5)
#    has E[T] = 21 exactly; for pert_model and sip_model use 1,000,000 simulations.
# 2. For n in (100, 1_000, 10_000): mc.coverage(sample_fn, interval_fn, truth, reps=400, rng=SEED)
#    for: Beta-PERT / CLT mean, SiP / CLT mean, Pareto chain / CLT mean, SiP / mc.quantile_ci(T, 0.9).
# 3. Display one table (rows = cases, columns = n), formatted as percentages.

TODO (Robin): interpret the coverage table. Which intervals keep their promise? Why does the CLT
interval fail with heavy tails (think about $\hat\sigma_n$)? What should we report instead?

### 5.4 Rare events: importance sampling

TODO (Robin): explain
* the relative error of plain Monte Carlo for a small probability $p$: $\sqrt{(1-p)/(np)}$;
* the importance-sampling identity $p = \mathbb E_g[\mathbf 1\{T > t\} f/g]$ (prove that it is unbiased);
* the two proposals ("tilt all tasks" and "tilt one task", $q_j \propto r_j^\theta$) and their weights;
* the "one big jump" principle: with heavy tails, $P(R_1 + \dots + R_k > x) \sim P(\max_i R_i > x)$.

In [ ]:
# P(T > 80 x plan): plain Monte Carlo versus importance sampling, same budget
# TODO (Robin):
# 1. Reference: 4,000,000 plain simulations in 4 batches of 1,000,000 (memory), p = share of T > 80 * plan.
# 2. For tilt in ["all", "one"] and theta in [0, 0.5, 1, 1.5, 2, 3]:
#    est = mc.tail_probability_is(project, project.durations, ratios, t, theta, n=100_000, rng=SEED, tilt=tilt)
#    Table: estimate and relative error (95%) = est.radius / est.value, for each (theta, tilt).

TODO (Robin): interpret the table. Which tilt works and which fails? By how much does the best one
reduce the variance (compare relative errors squared)? Why does the one-big-jump principle explain it?

### 5.5 Model risk: Monte Carlo cannot fix a wrong model

TODO (Robin): one sentence: we assumed independent tasks; compare with one common ratio for all tasks.

In [ ]:
# Independent tasks versus one common ratio for all tasks
# TODO (Robin): U independent = rng.random((n, k)); U common = np.repeat(rng.random((n, 1)), k, axis=1).
# For each, T = project.completion_time(sip_model.ppf(U)); table of E[T], 90% date, P(on time at plan).

TODO (Robin): interpret (how much does P(on time) change? are both estimates precise? what does this
say about Monte Carlo error bars?).

### 5.6 Other limitations

TODO (Robin): 2-3 bullets: transferability (SiP tasks vs PSPLIB networks, DSLIB as a real-data check),
rounded reporting in SiP (29% exactly on estimate), resources ignored.

---
## Conclusion

TODO (Robin, then all three of us): answer the research question in 3 short paragraphs with our numbers:
can we trust Monte Carlo (for which quantities)? when does it break down? what should a project manager
announce?

**References:** see `SOURCES.md` in the repository.